# Search for spliceosome complex component homologs in ciliates

This protocol is informed by https://onlinelibrary.wiley.com/doi/10.1111/jeu.12844 and https://www.ncbi.nlm.nih.gov/pmc/articles/PMC10158995/. However instead of a manually curated list of spliceosomal components, we pull reviewed annotations directly from Uniprot and OrthoDB through their SPARQL endpoint and/or API. This has the advantage of being faster and less error prone, and more reproducible. Disadvantage is that we cannot include unpublished data.

 * From Uniprot, get the list of proteins annotated with the GO term GO:0005681 (spliceosomal complex) in yeast, the model organism for most studies on the spliceosome, and where the annotation has been reviewed (Swissprot). Because annotations are inconsistent and patchy in most organisms, we choose to build a shortlist based on yeast proteins and then search for orthologs in our species of interest, rather than relying on the annotations in our species of interest.
 * Get the associated OrthoDB orthogroup for each protein.
 * Find the orthologs annotated by OrthoDB from ciliate genomes. NB: only seven ciliate genomes are included in OrthoDB:
     * Oxytricha trifallax
     * Pseudocohnilembus persalinus
     * Paramecium tetraurelia strain d4-2
     * Ichthyophthirius multifiliis
     * Stylonychia lemnae
     * Stentor coeruleus
     * Tetrahymena thermophila SB210
 * Search for homologs in Mesodinium rubrum (TODO)
     * Map predicted proteins to OrthoDB with orthomapper, and see if there are any hits to the orthogroups of interest
     * De novo ortholog clustering with OrthoFinder, and see whether the proteins of interest are clustered with those identified in other ciliate species
     * Blastx searches of the genome nucleotide sequence, in case the proteins were missed by gene prediction errors. How to distinguish against false positives? We could compare the E-values of hits against other proteins in the query genome which are positively grouped with a Mesodinium ortholog by OrthoFinder or OrthoMapper.
     * Search Mesodinium predicted proteins for any Pfam domains that are also found in spliceosomal components, based on the Pfam domains in ciliate spliceosomal components.

How did Wong et al. search for homologs to spliceosomal proteins in Porphyridium purpureum? (see their Supplementary Methods)

 * Reciprocal-best-hit blastp with spliceosomal proteins from human, yeast, Cyanidioschyzon merolae, and Arabidopsis
 * Filtered reciprocal-best-hit blastp results: >25% identity, >50% length coverage, E-value <1e-6
 * To identify more divergent homologs, related cutoffs: >20% identity, >??% length coverage
 * Repeated reciprocal-best-hit search with blastx and tblastn, in case some genes were not correctly predicted in the target genome

Comments to the above:

 * The set of query sequences used for reciprocal-best-hit should be the whole predicted proteome, rather than a shortlist of proteins, to have meaningful E-values, I think.
 * Reciprocal best hit procedure is essentially ortholog-clustering 'lite'. If we use Orthofinder or Orthomapper it should be equivalent

SPARQL endpoints and other resources

* UniProt SPARQL: https://sparql.uniprot.org/sparql
* OrthoDB SPARQL: https://sparql.orthodb.org/ (I could not find documentation aside from their example queries, unclear how some aspects are designed)
* UniProt SPARQL tutorials: https://github.com/sib-swiss/sparql-training
* UniProt SPARQL documentation: https://sparql.uniprot.org/.well-known/void
* Review on pre-mRNA splicing: 10.1002/9780470015902.a0005037.pub3

In [2]:
import json

In [3]:
from SPARQLWrapper import SPARQLWrapper, JSON

In [4]:
uniprot = SPARQLWrapper("https://sparql.uniprot.org/sparql")

In [5]:
yeast_spliceosome_components_query = """
PREFIX taxon: <http://purl.uniprot.org/taxonomy/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX up: <http://purl.uniprot.org/core/>
PREFIX obo: <http://purl.obolibrary.org/obo/>
PREFIX odbgroup: <http://purl.orthodb.org/odbgroup/> 
SELECT DISTINCT ?protein ?genename ?taxon ?sciname ?orthogroup 
WHERE {
  ?protein a up:Protein .
  ?protein up:organism ?taxon.
  ?taxon rdfs:subClassOf taxon:4932 . # Saccharomyces cerevisiae
  ?taxon up:scientificName ?sciname .
  ?protein up:encodedBy ?gene .
  ?protein up:reviewed true .
  ?protein up:classifiedWith ?goterm .
  ?goterm rdfs:subClassOf* obo:GO_0005681 . # spliceosomal complex 
  ?gene skos:prefLabel ?genename .
  ?protein rdfs:seeAlso ?orthogroup .
  FILTER (STRSTARTS(STR(?orthogroup), STR(odbgroup:))) # OrthoDB annotation 
} ORDER BY ?genename 
"""

In [6]:
uniprot.setQuery(yeast_spliceosome_components_query)

uniprot.setReturnFormat(JSON)

yeast_spliceosome_results = uniprot.query().convert()

In [1]:
# with open("yeast_spliceosome_results.json", "w") as fh:
#    json.dump(yeast_spliceosome_results, fh, indent=4)

In [7]:
len(yeast_spliceosome_results['results']['bindings'])

101

In [8]:
gene2ortholog = {
    i['genename']['value'] : {
        'protein' : i['protein']['value'], 
        'odbgroup' : i['orthogroup']['value'],
    }
  for i in yeast_spliceosome_results['results']['bindings']
}

In [9]:
gene2ortholog.keys()

dict_keys(['BRR1', 'BRR2', 'BUD13', 'BUD31', 'CBC2', 'CDC40', 'CEF1', 'CLF1', 'CUS1', 'CUS2', 'CWC15', 'CWC2', 'CWC21', 'CWC22', 'CWC23', 'CWC24', 'CWC25', 'CWC27', 'DIB1', 'DRN1', 'ECM2', 'FAL1', 'HSH155', 'HSH49', 'IST3', 'ISY1', 'JIP4', 'LEA1', 'LIN1', 'LOT5', 'LSM2', 'LSM3', 'LSM4', 'LSM5', 'LSM6', 'LSM7', 'LSM8', 'LUC7', 'MAK31', 'MSL1', 'MSL5', 'MUD1', 'MUD2', 'NAM8', 'NTC20', 'NTR2', 'PML1', 'PRP11', 'PRP16', 'PRP18', 'PRP19', 'PRP2', 'PRP21', 'PRP22', 'PRP24', 'PRP28', 'PRP3', 'PRP31', 'PRP38', 'PRP39', 'PRP4', 'PRP40', 'PRP42', 'PRP43', 'PRP45', 'PRP46', 'PRP5', 'PRP6', 'PRP8', 'PRP9', 'RDS3', 'RIM4', 'RSE1', 'SAD1', 'SLU7', 'SMB1', 'SMD1', 'SMD2', 'SMD3', 'SME1', 'SMX2', 'SMX3', 'SNP1', 'SNT309', 'SNU114', 'SNU13', 'SNU23', 'SNU56', 'SNU66', 'SNU71', 'SPP2', 'SPP381', 'SPP382', 'STO1', 'SUB2', 'SYF1', 'SYF2', 'URN1', 'YHC1', 'YJU2', 'YSF3'])

In [10]:
gene2ortholog['SYF1']

{'protein': 'http://purl.uniprot.org/uniprot/Q04048',
 'odbgroup': 'http://purl.orthodb.org/odbgroup/1130725at2759'}

In [11]:
orthodb = SPARQLWrapper("https://sparql.orthodb.org/sparql")

In [12]:
ortholog_ciliate_hits = {}

In [13]:
for gene in gene2ortholog:
    odbgroup = gene2ortholog[gene]['odbgroup'].split("/")[-1] # strip prefix
    ortholog_query = """
    PREFIX odb: <http://purl.orthodb.org/>
    PREFIX og: <http://purl.orthodb.org/odbgroup/>
    
    SELECT DISTINCT ?gene ?description ?sciname ?taxon
    where {
    ?taxon rdfs:subClassOf+/up:scientificName "Ciliophora" .
    ?gene a odb:Gene;
          odb:description ?description;
          up:organism/a ?taxon .
    ?taxon up:scientificName ?sciname .
    ?gene odb:memberOf+ og:%s.
    } ORDER BY ?sciname
    """ % (odbgroup, )
    orthodb.setQuery(ortholog_query)
    orthodb.setReturnFormat(JSON)
    results = orthodb.query().convert()
    ortholog_ciliate_hits[gene] = results['results']['bindings']

Which spliceosomal components do not have an ortholog representative in ciliate genomes?

In [14]:
[gene for gene in ortholog_ciliate_hits if len(ortholog_ciliate_hits[gene]) == 0]

['BRR1',
 'CWC25',
 'JIP4',
 'LIN1',
 'LOT5',
 'LUC7',
 'MAK31',
 'NTC20',
 'NTR2',
 'PRP24',
 'PRP38',
 'RIM4',
 'SNT309',
 'SNU23',
 'SNU56',
 'SNU66',
 'SNU71',
 'SPP381',
 'SYF2',
 'URN1']

Should be interpreted with caution, as there may be related proteins/orthologs with the same function in ciliates but which are not in the same cluster as the yeast protein. For example, there are multiple ortholog groups with LUC7 related domains in OrthoDB, only one of which is in the yeast shortlist (not represented in ciliates), but others are found in ciliates.

In [15]:
import pandas as pd

In [16]:
ortholog_ciliate_hits_df = pd.DataFrame(
    [
        {'genename' : genename, 'gene' : rec['gene']['value'], 'desc' : rec['description']['value'], 'sciname' : rec['sciname']['value']}
        for genename in ortholog_ciliate_hits
        for rec in ortholog_ciliate_hits[genename]# if len(ortholog_ciliate_hits[genename]) > 0
    ]
)

In [17]:
ortholog_ciliate_hits_df.groupby(['genename','sciname'])[['genename','sciname']].value_counts().reset_index().pivot(index='genename',columns='sciname',values='count').fillna(0)

sciname,Ichthyophthirius multifiliis,Oxytricha trifallax,Paramecium tetraurelia strain d4-2,Pseudocohnilembus persalinus,Stentor coeruleus,Stylonychia lemnae,Tetrahymena thermophila SB210
genename,,,,,,,
BRR2,3.0,2.0,4.0,2.0,2.0,2.0,2.0
BUD13,1.0,1.0,2.0,1.0,1.0,1.0,1.0
BUD31,1.0,1.0,2.0,1.0,2.0,1.0,1.0
CBC2,1.0,2.0,2.0,1.0,1.0,2.0,1.0
CDC40,1.0,1.0,2.0,1.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...
SUB2,7.0,9.0,15.0,4.0,12.0,9.0,4.0
SYF1,1.0,1.0,2.0,1.0,1.0,1.0,1.0
YHC1,0.0,1.0,2.0,1.0,3.0,0.0,1.0


In [18]:
# How many orthologs represented per genome (presence/absence only, not accounting for number of copies)
ortholog_ciliate_hits_df.groupby(['sciname','genename']).size().reset_index().groupby('sciname').size()

sciname
Ichthyophthirius multifiliis          62
Oxytricha trifallax                   76
Paramecium tetraurelia strain d4-2    76
Pseudocohnilembus persalinus          67
Stentor coeruleus                     66
Stylonychia lemnae                    70
Tetrahymena thermophila SB210         77
dtype: int64

In [19]:
published_shortlist = pd.read_csv("spliceosome_shortlist.tsv",sep="\t")

How many of the yeast spliceosomal proteins we pulled from Uniprot are not in the published shortlist of 71 spliceosomal proteins from Black et al. (https://doi.org/10.1261%2Frna.079273.122)?

In [20]:
len([i for i in gene2ortholog if i not in list(published_shortlist['genename'])])

34

Which gene products named in the Black et al. shortlist are not in our list pulled from Uniprot?

In [21]:
[i for i in list(published_shortlist['genename']) if i not in gene2ortholog]

['AAR2', 'SNU40', 'SNY309', 'PRP17']

Only one of these is truly not in our shortlist, namely AAR2 (https://www.uniprot.org/uniprotkb/P32357). I'm not sure why it is not annotated with GO:0005681.

To the remainder:
 * PRP17 is a synonym for CDC40 https://www.uniprot.org/uniprotkb/P40968
 * SNU40 may be a synonym for LIN1, have to check https://www.uniprot.org/uniprotkb/P38852
 * SNY309 was not found by keyword search of Uniprot, and is probably a typo for SNT309

In [38]:
published_shortlist.query('group=="U1"')

,genename,group
0,LUC7,U1
1,MUD1,U1
2,NAM8,U1
3,PRP39,U1
4,SNP1,U1
5,YHC1,U1


In [100]:
ortholog_ciliate_hits_df

,genename,gene,desc,sciname
0,BRR2,http://purl.orthodb.org/odbgene/5932_0_000698,hypothetical protein,Ichthyophthirius multifiliis
1,BRR2,http://purl.orthodb.org/odbgene/5932_0_000692,hypothetical protein,Ichthyophthirius multifiliis
2,BRR2,http://purl.orthodb.org/odbgene/5932_0_000800,hypothetical protein,Ichthyophthirius multifiliis
3,BRR2,http://purl.orthodb.org/odbgene/1172189_1_000cbd,J9IZ88_9SPIT,Oxytricha trifallax
4,BRR2,http://purl.orthodb.org/odbgene/1172189_1_0046c9,J9IGS8_9SPIT,Oxytricha trifallax
...,...,...,...,...
1014,YSF3,http://purl.orthodb.org/odbgene/5932_0_0012f4,hypothetical protein,Ichthyophthirius multifiliis
1015,YSF3,http://purl.orthodb.org/odbgene/412030_0_0076f0,hypothetical protein,Paramecium tetraurelia strain d4-2
1016,YSF3,http://purl.orthodb.org/odbgene/266149_1_000564,A0A0V0QHA1_PSEPJ,Pseudocohnilembus persalinus
1017,YSF3,http://purl.orthodb.org/odbgene/5963_1_0027cc,Splicing factor subunit,Stentor coeruleus


In [40]:
ortholog_ciliate_hits_df.query('genename == "CWC22"')

,genename,gene,desc,sciname
114,CWC22,http://purl.orthodb.org/odbgene/5932_0_000dfa,"pre-mRNA-splicing factor cwc-22, putative",Ichthyophthirius multifiliis
115,CWC22,http://purl.orthodb.org/odbgene/1172189_1_00365b,J9IRL9_9SPIT,Oxytricha trifallax
116,CWC22,http://purl.orthodb.org/odbgene/412030_0_004752,hypothetical protein,Paramecium tetraurelia strain d4-2
117,CWC22,http://purl.orthodb.org/odbgene/412030_0_00929e,hypothetical protein,Paramecium tetraurelia strain d4-2
118,CWC22,http://purl.orthodb.org/odbgene/266149_1_001a34,A0A0V0QSI4_PSEPJ,Pseudocohnilembus persalinus
119,CWC22,http://purl.orthodb.org/odbgene/5963_1_003db7,MI domain-containing protein,Stentor coeruleus
120,CWC22,http://purl.orthodb.org/odbgene/5949_1_0006be,Contig5904.g6329,Stylonychia lemnae
121,CWC22,http://purl.orthodb.org/odbgene/312017_0_002e1f,Q23JX2_TETTS,Tetrahymena thermophila SB210


## Get sequences for each ortholog

In [28]:
import requests

In [29]:
fasta_url = "https://data.orthodb.org/current/fasta"

In [30]:
req = requests.get(url=fasta_url, params={"id" : "57056at2759", "species" : "5878"})

In [36]:
for g in gene2ortholog:
    odbgroup = gene2ortholog[g]['odbgroup'].split("/")[-1]
    if len(ortholog_ciliate_hits[g]) > 0:
        req = requests.get(url=fasta_url, params={"id" : odbgroup, "species" : "5878"})
        if req.ok:
            fn = f"spliceosome_orthologs/{g}.fasta"
            with open(fn, "w") as fh:
                fh.write(req.text)

## Ribosomal proteins

Repeat comparable procedure for ribosomal proteins in ciliates. Specify cytoplasmic location to avoid picking up mitochondrial ribosomal proteins.

In [41]:
yeast_ribosomal_proteins_query = """PREFIX taxon: <http://purl.uniprot.org/taxonomy/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX up: <http://purl.uniprot.org/core/>
PREFIX obo: <http://purl.obolibrary.org/obo/>
PREFIX odbgroup: <http://purl.orthodb.org/odbgroup/>
PREFIX keywords: <http://purl.uniprot.org/keywords/>
PREFIX locations: <http://purl.uniprot.org/locations/>

SELECT DISTINCT ?protein ?genename ?taxon ?sciname ?orthogroup
WHERE {
  ?protein a up:Protein .
  ?protein up:organism ?taxon.
  ?taxon rdfs:subClassOf taxon:4932 . # Saccharomyces cerevisiae
  ?taxon up:scientificName ?sciname .
  ?protein up:encodedBy ?gene .
  ?protein up:reviewed true .
  ?protein up:classifiedWith keywords:689. # keyword "ribosomal protein"
  ?protein up:annotation ?annot.
  ?annot up:locatedIn/up:cellularComponent locations:86 . # cytoplasmic 
  ?gene skos:prefLabel ?genename .
  ?protein rdfs:seeAlso ?orthogroup .
  FILTER (STRSTARTS(STR(?orthogroup), STR(odbgroup:))) # OrthoDB annotation 
} ORDER BY ?genename 
"""

In [42]:
uniprot.setQuery(yeast_ribosomal_proteins_query)
uniprot.setReturnFormat(JSON)
yeast_ribosomal_protein_results = uniprot.query().convert()

In [44]:
rp_gene2ortholog = {
    i['genename']['value'] : {
        'protein' : i['protein']['value'], 
        'odbgroup' : i['orthogroup']['value'],
    }
  for i in yeast_ribosomal_protein_results['results']['bindings']
}

In [45]:
len(rp_gene2ortholog)

137

In [46]:
%%bash
mkdir -p rp_orthologs

In [48]:
for g in rp_gene2ortholog:
    odbgroup = rp_gene2ortholog[g]['odbgroup'].split("/")[-1]
    req = requests.get(url=fasta_url, params={"id" : odbgroup, "species" : "5878"})
    if req.ok:
        fn = f"rp_orthologs/{odbgroup}.ciliates.fasta"
        with open(fn, "w") as fh:
            fh.write(req.text)

## Tricarboxylic acid cycle components

In [51]:
yeast_tca_components_query = """
PREFIX taxon: <http://purl.uniprot.org/taxonomy/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX up: <http://purl.uniprot.org/core/>
PREFIX obo: <http://purl.obolibrary.org/obo/>
PREFIX odbgroup: <http://purl.orthodb.org/odbgroup/> 
SELECT DISTINCT ?protein ?genename ?taxon ?sciname ?orthogroup 
WHERE {
  ?protein a up:Protein .
  ?protein up:organism ?taxon.
  ?taxon rdfs:subClassOf taxon:4932 . # Saccharomyces cerevisiae
  ?taxon up:scientificName ?sciname .
  ?protein up:encodedBy ?gene .
  ?protein up:reviewed true .
  ?protein up:classifiedWith ?goterm .
  ?goterm rdfs:subClassOf* obo:GO_0006099 . # TCA cycle
  ?gene skos:prefLabel ?genename .
  ?protein rdfs:seeAlso ?orthogroup .
  FILTER (STRSTARTS(STR(?orthogroup), STR(odbgroup:))) # OrthoDB annotation 
} ORDER BY ?genename 
"""

In [52]:
uniprot.setQuery(yeast_tca_components_query)
uniprot.setReturnFormat(JSON)
yeast_tca_components_results = uniprot.query().convert()

In [53]:
tca_gene2ortholog = {
    i['genename']['value'] : {
        'protein' : i['protein']['value'], 
        'odbgroup' : i['orthogroup']['value'],
    }
  for i in yeast_tca_components_results['results']['bindings']
}

In [54]:
%%bash
mkdir -p tca_orthologs

In [55]:
for g in tca_gene2ortholog:
    odbgroup = tca_gene2ortholog[g]['odbgroup'].split("/")[-1]
    req = requests.get(url=fasta_url, params={"id" : odbgroup, "species" : "5878"})
    if req.ok:
        fn = f"tca_orthologs/{odbgroup}.ciliates.fasta"
        with open(fn, "w") as fh:
            fh.write(req.text)

## Which BUSCO markers are spliceosome-related?

Find ortholog groups at Alveolata level corresponding to these spliceosome related genes in Ciliophora

Unfortunately there does not appear to be a way to search/access the ortholog group hierarchy data through the API. Instead, I find the genes from ciliates that are in ortholog groups relevant to spliceosome, and then pull out the relevant ortholog groups at the Alveolata level, to see how many of them appear in the BUSCO marker list for Alveolata (ODB v10).

There is also no mapping between v10 and v11 of OrthoDB, and both the ortholog IDs and gene IDs are not stable between OrthoDB releases, so we have to manually map them somehow ... 

In [106]:
spliceosome_ciliate_gene_ids = list(ortholog_ciliate_hits_df['gene'].str.split("/", expand=True)[4])

In [107]:
spliceosome_ciliate_gene_ids = [i.split("_") for i in spliceosome_ciliate_gene_ids]

In [110]:
spliceosome_ciliate_gene_ids = [i[0] + "_" + i[1] + ":" + i[2] for i in spliceosome_ciliate_gene_ids]

In [142]:
spliceosome_ciliate_gene_ids[0:10]

['5932_0:000698',
 '5932_0:000692',
 '5932_0:000800',
 '1172189_1:000cbd',
 '1172189_1:0046c9',
 '412030_0:000f4c',
 '412030_0:004f7b',
 '412030_0:0014cb',
 '412030_0:004ee4',
 '266149_1:002d39']

In [111]:
import gzip

In [145]:
%%bash
zcat ../db/odb11v0_gene_xrefs.tab.gz | head

100_0:000000	GO:0000166	GOterm
100_0:000000	GO:0005524	GOterm
100_0:000000	IPR017871	InterPro
100_0:000000	IPR003439	InterPro
100_0:000000	IPR027417	InterPro
100_0:000000	IPR003593	InterPro
100_0:000000	WP_131833652.1	NCBIproteinAcc
100_0:000000	A0A4R1I8Y6	UniProt
100_0:000001	GO:0046872	GOterm
100_0:000001	IPR026045	InterPro


In [149]:
spliceosome_ciliate_odb11_id2ncbi = {}
counter = 0
with gzip.open("../db/odb11v0_gene_xrefs.tab.gz", "rt") as fh:
    for line in fh:
        [geneid, xref, xrefdb] = line.rstrip().split("\t")
        if xrefdb == "NCBIproteinAcc" and geneid in spliceosome_ciliate_gene_ids:
            spliceosome_ciliate_odb11_id2ncbi[geneid] = xref
            counter += 1
            if counter % 100 == 0:
                print(str(counter) + " found...")

100 found...
200 found...
300 found...
400 found...
500 found...
600 found...
700 found...
800 found...


In [151]:
spliceosome_ciliate_odb11_ncbi2id = {
    spliceosome_ciliate_odb11_id2ncbi[i] : i for i in spliceosome_ciliate_odb11_id2ncbi
}

In [152]:
%%bash
zcat ../db/odb11v0_gene_xrefs.tab.gz | head

100_0:000000	GO:0000166	GOterm
100_0:000000	GO:0005524	GOterm
100_0:000000	IPR017871	InterPro
100_0:000000	IPR003439	InterPro
100_0:000000	IPR027417	InterPro
100_0:000000	IPR003593	InterPro
100_0:000000	WP_131833652.1	NCBIproteinAcc
100_0:000000	A0A4R1I8Y6	UniProt
100_0:000001	GO:0046872	GOterm
100_0:000001	IPR026045	InterPro


In [159]:
spliceosome_ciliate_odb10_to_odb11 = {}
counter = 0
with gzip.open("../db/odb10v1_gene_xrefs.tab.gz", "rt") as fh:
    for line in fh:
        [odb10id, xref, xrefdb] = line.rstrip().split("\t")
        if xrefdb == "NCBIproteinAcc" and xref in spliceosome_ciliate_odb11_ncbi2id:
            counter += 1
            odb11id = spliceosome_ciliate_odb11_ncbi2id[xref]
            spliceosome_ciliate_odb10_to_odb11[odb10id] = odb11id
            if counter % 100 == 0:
                print(str(counter) + " found...")

100 found...
200 found...
300 found...
400 found...
500 found...
600 found...
700 found...
800 found...


In [160]:
len(spliceosome_ciliate_odb10_to_odb11)

889

In [161]:
len(spliceosome_ciliate_odb11_ncbi2id)

889

In [163]:
# find ortholog clusters from odb10 used as BUSCO markers for clade Alveolata
# based on this shortlist of ciliate spliceosome-related proteins

spliceosome_ciliate_odb10_busco_related = {}
with open("../db/busco/lineages/alveolata_odb10/info/ogs.id.info", "r") as fh:
    for line in fh:
        [geneid, orthogroup] = line.rstrip().split("\t")
        if geneid in spliceosome_ciliate_odb10_to_odb11:
            spliceosome_ciliate_odb10_busco_related[geneid] = orthogroup

In [164]:
len(spliceosome_ciliate_odb10_busco_related)

119

In [167]:
len(set(list(spliceosome_ciliate_odb10_busco_related.values())))

18

In [168]:
set(list(spliceosome_ciliate_odb10_busco_related.values()))

{'11233at33630',
 '12825at33630',
 '15452at33630',
 '1636at33630',
 '16788at33630',
 '20963at33630',
 '22490at33630',
 '22819at33630',
 '24978at33630',
 '25715at33630',
 '2846at33630',
 '30838at33630',
 '3134at33630',
 '3330at33630',
 '34124at33630',
 '4205at33630',
 '450at33630',
 '7531at33630'}

In [171]:
inlist = """6394at33630
1636at33630
20963at33630
15452at33630
22490at33630
4205at33630
24978at33630
30838at33630
7531at33630
3134at33630
12825at33630
2846at33630
21382at33630
450at33630
16788at33630
34124at33630
25715at33630"""
inlist = inlist.split("\n")

In [173]:
[i for i in inlist if i not in set(list(spliceosome_ciliate_odb10_busco_related.values()))]

['6394at33630', '21382at33630']

6394at33630
Heat shock protein 70 family

21382at33630
RNA recognition motif domain

In [174]:
[i for i in set(list(spliceosome_ciliate_odb10_busco_related.values())) if i not in inlist]

['3330at33630', '22819at33630', '11233at33630']

3330at33630
Translation elongation factor EFG/EF2, domain IV

22819at33630
Zinc finger, CCCH-type

11233at33630
Helicase, C-terminal

Not much difference in the total number of orthologs related to spliceosome between ODB10 and ODB11, although there are individual differences in some ortholog groups.

The initial assessment that about 10% of BUSCO markers for Alveolata are spliceosome-related still stands.

In [129]:
import re

In [134]:
from collections import defaultdict

In [133]:
re.match(r"(.+)at(.+)", '8103at33630').groups()

('8103', '33630')

In [135]:
busco_alveolata_gene2og = defaultdict(dict)
with open("../db/busco/lineages/alveolata_odb10/info/ogs.id.info", "r") as fh:
    for line in fh:
        [gene_id, orthogroup] = line.rstrip().split("\t")
        ogid, taxid = re.match(r"(.+)at(.+)", orthogroup).groups()
        busco_alveolata_gene2og[gene_id][taxid] = orthogroup

In [139]:
%%bash
zcat ../db/odb10v1_OG2genes.tab.gz | head

0at100953	111167_1:000002
0at100953	1123958_1:00001c
0at100953	114102_1:000001
0at100953	114430_1:000014
0at100953	1193422_1:000016
0at100953	129875_1:000005
0at100953	129951_1:000004
0at100953	129953_1:00000c
0at100953	129956_1:000005
0at100953	130308_1:000003


In [140]:
with gzip.open("../db/odb10v1_OG2genes.tab.gz", "rt") as fh:
    for line in fh:
        orthogroup, geneid = line.rstrip().split("\t")
        if geneid in busco_alveolata_gene2og:
            ogid, taxid = re.match(r"(.+)at(.+)", orthogroup).groups()
            busco_alveolata_gene2og[geneid][taxid] = orthogroup